> ⚠️ First, you need to **select the correct environment/kernel** to run the following notebook. Please select the kernel named **"Python (CSBDeep)"** ⚠️

You should have perform preprocessing of data using Notebook #2 [2D-CARE_training.ipynb](2D-CARE_training.ipynb) and have a noisy image to denoise never seen by the network.

# 2D-CARE Prediction

This notebook allows you to make a prediction using the 2D-CARE model trained with the previous notebook.

## Importing libraries & functions
In the following section, we will load Python librairies and functions required. Run the next cell to load librairies and dependencies:

In [ ]:
# Run to load libraries

import os
import sys
import re

import time as time
import numpy as np

from pathlib import Path
from ipywidgets import widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

sys.path.append(os.path.abspath('..'))

from care.prediction import do_denoising_3D

## Load trained model
First, you need to load the training model. A training model can be used to restore images from the same training batch. The model is located in the modele/name_of_model/config.json folder (CARE's default directory structure).

Run the cell below to load the model and the image to denoise.

In [ ]:
fc_image = FileChooser('.')
fc_image.title = '<b>Select image to denoise (.tif)</b>'
fc_image.filter_pattern = ['*.tif', '*.tiff']

fc_model = FileChooser('.')
fc_model.title = '<b>Select trained model (eg: k3d2f64_0001)</b>'
fc_model.show_only_dirs = True

display(widgets.VBox([
    widgets.HTML("<h3>Loading data and model</h3>"),
    fc_model,
    fc_image
]))

Then, you can run the denoising process by executing the next cell. By default, the image is saved in the same folder as the noisy image.

In [ ]:
def check_model(nom_modele):
    pattern = r"k\d+.*d\d+"
    return bool(re.search(pattern, nom_modele, re.IGNORECASE))

if not fc_image.selected or not fc_model.selected:
    raise ValueError("Error : Select a model and an image first.")

image_path = Path(fc_image.selected)
output_path = image_path.with_name(f"{image_path.stem}_denoised_care{image_path.suffix}")

model_full_path = Path(fc_model.selected)
model_name = model_full_path.name       # eg: training_k3d1l64_0001
model_dir = model_full_path.parent      # eg: /Users/.../test_data/models

if not check_model(model_name):
    print(f"Warning : Model name '{model_name}' is not in the expected format.")

print(f"Image: {image_path}")
print(f"Saving name: {output_path}")
print(f"Model used: {model_name}")
print("Starting...")

start = time.time()

do_denoising_3D(
    input_path=str(image_path), 
    output_path=str(output_path), 
    model_name=model_name,
    basedir=str(model_dir) 
)

end = time.time()
print(f"Finished in {(end - start) / 60:.2f} minutes !")